# Master Model Evaluation - Official Test Set

**Purpose:** Evaluate all trained models on the official held-out test set  
**Test Set:** 968 images (patient-stratified, never seen during training)  
**Models:** ResNet-50 Baseline, SE-ResNet50, CBAM-ResNet50, ViT-B/16

## Evaluation Protocol

1. Discover all trained model checkpoints
2. Load best checkpoint for each model/seed combination
3. Evaluate on official test set
4. Compute comprehensive metrics (accuracy, F1, precision, recall, per-class)
5. Generate comparison tables and statistics

In [16]:
# IMPORTS
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from pathlib import Path
import numpy as np
import pandas as pd
import re
from collections import defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Imports successful")

Imports successful


In [17]:
# CONFIGURATION

ROOT = Path(r"C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training")

CHECKPOINT_DIR = ROOT / "Checkpoints"
TEST_PATH = ROOT / "Data_Kermany_OCT2017" / "test"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Test set path: {TEST_PATH}")
print(f"Device: {DEVICE}")
print("="*80)

CONFIGURATION
Checkpoint directory: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints
Test set path: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Data_Kermany_OCT2017\test
Device: cuda


In [18]:
# MODEL ARCHITECTURE DEFINITIONS

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel attention."""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class SEResNet50(nn.Module):
    """ResNet-50 with Squeeze-and-Excitation attention."""
    def __init__(self, num_classes=4, pretrained=False, reduction=16):
        super(SEResNet50, self).__init__()
        resnet = models.resnet50(pretrained=pretrained)
        
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        self.se1 = SEBlock(256, reduction)
        self.se2 = SEBlock(512, reduction)
        self.se3 = SEBlock(1024, reduction)
        self.se4 = SEBlock(2048, reduction)
        
        self.avgpool = resnet.avgpool
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.se1(x)
        x = self.layer2(x)
        x = self.se2(x)
        x = self.layer3(x)
        x = self.se3(x)
        x = self.layer4(x)
        x = self.se4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class ChannelAttention(nn.Module):
    """Channel attention for CBAM."""
    def __init__(self, channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        max_out = self.mlp(self.max_pool(x).view(b, c))
        out = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * out.expand_as(x)


class SpatialAttention(nn.Module):
    """Spatial attention for CBAM."""
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv(out))
        return x * out


class CBAM(nn.Module):
    """Convolutional Block Attention Module."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


class CBAMResNet50(nn.Module):
    """ResNet-50 with CBAM attention."""
    def __init__(self, num_classes=4, pretrained=False, reduction=16):
        super(CBAMResNet50, self).__init__()
        resnet = models.resnet50(pretrained=pretrained)
        
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        self.cbam1 = CBAM(256, reduction)
        self.cbam2 = CBAM(512, reduction)
        self.cbam3 = CBAM(1024, reduction)
        self.cbam4 = CBAM(2048, reduction)
        
        self.avgpool = resnet.avgpool
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.cbam1(x)
        x = self.layer2(x)
        x = self.cbam2(x)
        x = self.layer3(x)
        x = self.cbam3(x)
        x = self.layer4(x)
        x = self.cbam4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def create_model(model_type, num_classes=4):
    """Factory function to create models by type."""
    if model_type == 'resnet_baseline':
        model = models.resnet50(pretrained=False)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_type == 'se_resnet':
        model = SEResNet50(num_classes=num_classes, pretrained=False)
    elif model_type == 'cbam_resnet':
        model = CBAMResNet50(num_classes=num_classes, pretrained=False)
    elif model_type == 'vit_b16':
        model = vit_b_16(weights=None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    return model


print("Model architectures defined")

Model architectures defined


In [20]:
# PARAMETER COUNT FOR ALL ARCHITECTURES
# Run this cell AFTER Cell 3 (model definitions) has been executed

print("=" * 80)
print("MODEL PARAMETER COUNTS")
print("=" * 80)

model_types = ['resnet_baseline', 'se_resnet', 'cbam_resnet', 'vit_b16']
model_display_names = {
    'resnet_baseline': 'ResNet-50',
    'se_resnet': 'SE-ResNet50',
    'cbam_resnet': 'CBAM-ResNet50',
    'vit_b16': 'ViT-B/16'
}

print(f"\n{'Architecture':<20s} {'Total Params':>15s} {'Trainable':>15s} {'Rel. to Baseline':>18s}")
print("-" * 70)

baseline_params = None

for model_type in model_types:
    model = create_model(model_type, NUM_CLASSES)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    if baseline_params is None:
        baseline_params = total_params
        rel = "Baseline"
    else:
        pct_increase = ((total_params - baseline_params) / baseline_params) * 100
        rel = f"+{pct_increase:.1f}%"
    
    display_name = model_display_names[model_type]
    print(f"{display_name:<20s} {total_params:>15,d} {trainable_params:>15,d} {rel:>18s}")
    
    del model  # Free memory

print("-" * 70)
print("\nValues to update Table 5.8 in Chapter 5.")

MODEL PARAMETER COUNTS

Architecture            Total Params       Trainable   Rel. to Baseline
----------------------------------------------------------------------
ResNet-50                 23,516,228      23,516,228           Baseline
SE-ResNet50               24,212,548      24,212,548              +3.0%
CBAM-ResNet50             24,212,940      24,212,940              +3.0%
ViT-B/16                  85,801,732      85,801,732            +264.9%
----------------------------------------------------------------------

Values to update Table 5.8 in Chapter 5.


In [4]:
# CHECKPOINT DISCOVERY

def discover_checkpoints(checkpoint_dir, verbose=False):
    """
    Discover all model checkpoints and organize by serial/seed.
    
    Returns dict: {serial_number: {seed: {model_name, checkpoints}}}
    """
    checkpoint_dir = Path(checkpoint_dir)
    
    # Pattern: SERIAL_modelname_seedN_epochM_mode_timestamp.pth
    pattern = re.compile(r'^(\d+)_([a-z0-9_]+)_seed(\d+)_epoch(\d+)_(best|last|intermediate)_.*\.pth$')
    
    discovered = defaultdict(lambda: defaultdict(lambda: {'checkpoints': [], 'model_name': None}))
    
    checkpoint_files = list(checkpoint_dir.glob("*.pth"))
    
    if verbose:
        print(f"\nScanning {len(checkpoint_files)} checkpoint files...\n")
    
    for filepath in checkpoint_files:
        match = pattern.match(filepath.name)
        if match:
            serial = int(match.group(1))
            model_name = match.group(2)
            seed = int(match.group(3))
            epoch = int(match.group(4))
            mode = match.group(5)
            
            discovered[serial][seed]['model_name'] = model_name
            discovered[serial][seed]['checkpoints'].append({
                'path': filepath,
                'epoch': epoch,
                'mode': mode,
                'filename': filepath.name
            })
            
            if verbose:
                print(f"  Serial {serial:02d}, Seed {seed}, {model_name}, Epoch {epoch}, {mode}")
    
    # Convert to regular dict for cleaner output
    return {k: dict(v) for k, v in discovered.items()}


def find_best_checkpoint(checkpoints_dict, serial, seed):
    """
    Find the best checkpoint for a given serial/seed combination.
    Priority: best mode > last mode > highest epoch intermediate
    """
    if serial not in checkpoints_dict or seed not in checkpoints_dict[serial]:
        return None
    
    checkpoints = checkpoints_dict[serial][seed]['checkpoints']
    
    # Priority 1: best checkpoint
    best_ckpts = [c for c in checkpoints if c['mode'] == 'best']
    if best_ckpts:
        return max(best_ckpts, key=lambda x: x['epoch'])
    
    # Priority 2: last checkpoint
    last_ckpts = [c for c in checkpoints if c['mode'] == 'last']
    if last_ckpts:
        return max(last_ckpts, key=lambda x: x['epoch'])
    
    # Priority 3: highest epoch intermediate
    if checkpoints:
        return max(checkpoints, key=lambda x: x['epoch'])
    
    return None


# Discover all checkpoints
print("="*80)
print("DISCOVERING CHECKPOINTS")
print("="*80)

all_checkpoints = discover_checkpoints(CHECKPOINT_DIR, verbose=True)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

for serial in sorted(all_checkpoints.keys()):
    for seed in sorted(all_checkpoints[serial].keys()):
        info = all_checkpoints[serial][seed]
        model_name = info['model_name']
        num_checkpoints = len(info['checkpoints'])
        best = find_best_checkpoint(all_checkpoints, serial, seed)
        
        print(f"Serial {serial:02d} | Seed {seed} | {model_name:20s} | "
              f"{num_checkpoints:2d} checkpoints | Best: Epoch {best['epoch'] if best else 'N/A'}")

print("="*80)

DISCOVERING CHECKPOINTS

Scanning 511 checkpoint files...

  Serial 01, Seed 42, resnet_baseline, Epoch 10, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 11, best
  Serial 01, Seed 42, resnet_baseline, Epoch 13, best
  Serial 01, Seed 42, resnet_baseline, Epoch 14, best
  Serial 01, Seed 42, resnet_baseline, Epoch 15, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 1, best
  Serial 01, Seed 42, resnet_baseline, Epoch 20, best
  Serial 01, Seed 42, resnet_baseline, Epoch 20, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 21, best
  Serial 01, Seed 42, resnet_baseline, Epoch 25, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 27, best
  Serial 01, Seed 42, resnet_baseline, Epoch 28, best
  Serial 01, Seed 42, resnet_baseline, Epoch 2, best
  Serial 01, Seed 42, resnet_baseline, Epoch 30, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 35, intermediate
  Serial 01, Seed 42, resnet_baseline, Epoch 38, best
  Serial 01, Seed 42, resnet_ba

In [5]:
# TEST SET LOADING

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = ImageFolder(root=str(TEST_PATH), transform=test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print("="*80)
print("TEST SET LOADED")
print("="*80)
print(f"Total images: {len(test_dataset)}")
print(f"Batches: {len(test_loader)}")
print(f"Classes: {test_dataset.classes}")
print("="*80)

TEST SET LOADED
Total images: 968
Batches: 31
Classes: ['CNV', 'DME', 'DRUSEN', 'NORMAL']


In [6]:
# EVALUATION FUNCTION

def evaluate_model(model, test_loader, device):
    """
    Evaluate model on test set and compute comprehensive metrics.
    
    Returns dict with accuracy, F1, precision, recall, and per-class metrics.
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating", leave=False):
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Overall metrics
    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1_macro = f1_score(all_labels, all_preds, average='macro') * 100
    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0) * 100
    recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100
    
    # Per-class metrics
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0) * 100
    precision_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0) * 100
    recall_per_class = recall_score(all_labels, all_preds, average=None, zero_division=0) * 100
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_per_class': f1_per_class,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'labels': all_labels
    }


print("Evaluation function defined")

Evaluation function defined


In [7]:
# EVALUATE ALL MODELS

print("="*80)
print("EVALUATING ALL MODELS")
print("="*80)

results = []

for serial in sorted(all_checkpoints.keys()):
    for seed in sorted(all_checkpoints[serial].keys()):
        info = all_checkpoints[serial][seed]
        model_name = info['model_name']
        
        best_checkpoint = find_best_checkpoint(all_checkpoints, serial, seed)
        
        if best_checkpoint is None:
            print(f"\n⚠️  No checkpoint found for Serial {serial:02d}, Seed {seed}")
            continue
        
        print(f"\n{'='*80}")
        print(f"Serial {serial:02d} | Seed {seed} | {model_name.upper()}")
        print(f"Checkpoint: {best_checkpoint['filename']}")
        print(f"Epoch: {best_checkpoint['epoch']} | Mode: {best_checkpoint['mode']}")
        print(f"{'='*80}")
        
        # Create model
        model = create_model(model_name, NUM_CLASSES)
        
        # Load checkpoint
        checkpoint = torch.load(best_checkpoint['path'], map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(DEVICE)
        
        # Evaluate
        metrics = evaluate_model(model, test_loader, DEVICE)
        
        # Store results
        result = {
            'serial': serial,
            'seed': seed,
            'model_name': model_name,
            'epoch': best_checkpoint['epoch'],
            'checkpoint_mode': best_checkpoint['mode'],
            **metrics
        }
        results.append(result)
        
        # Print results
        print(f"\nTest Set Results:")
        print(f"  Accuracy:  {metrics['accuracy']:.2f}%")
        print(f"  F1-Score:  {metrics['f1_macro']:.2f}%")
        print(f"  Precision: {metrics['precision_macro']:.2f}%")
        print(f"  Recall:    {metrics['recall_macro']:.2f}%")
        
        print(f"\nPer-Class F1-Scores:")
        for i, class_name in enumerate(CLASS_NAMES):
            print(f"  {class_name:8s}: {metrics['f1_per_class'][i]:.2f}%")

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)

EVALUATING ALL MODELS

Serial 01 | Seed 42 | RESNET_BASELINE
Checkpoint: 01_resnet_baseline_seed42_epoch46_best_20260114_001728.pth
Epoch: 46 | Mode: best



Test Set Results:
  Accuracy:  99.07%
  F1-Score:  99.07%
  Precision: 99.09%
  Recall:    99.07%

Per-Class F1-Scores:
  CNV     : 98.37%
  DME     : 99.79%
  DRUSEN  : 98.33%
  NORMAL  : 99.79%

Serial 02 | Seed 42 | SE_RESNET
Checkpoint: 02_se_resnet_seed42_epoch47_best_20260114_020922.pth
Epoch: 47 | Mode: best



Test Set Results:
  Accuracy:  98.24%
  F1-Score:  98.25%
  Precision: 98.34%
  Recall:    98.24%

Per-Class F1-Scores:
  CNV     : 96.80%
  DME     : 99.79%
  DRUSEN  : 96.60%
  NORMAL  : 99.79%

Serial 03 | Seed 84 | RESNET_BASELINE
Checkpoint: 03_resnet_baseline_seed84_epoch50_best_20260114_101157.pth
Epoch: 50 | Mode: best



Test Set Results:
  Accuracy:  98.86%
  F1-Score:  98.87%
  Precision: 98.90%
  Recall:    98.86%

Per-Class F1-Scores:
  CNV     : 97.98%
  DME     : 99.79%
  DRUSEN  : 97.90%
  NORMAL  : 99.79%

Serial 04 | Seed 42 | SE_RESNET
Checkpoint: 04_se_resnet_seed42_epoch47_best_20260114_115805.pth
Epoch: 47 | Mode: best



Test Set Results:
  Accuracy:  98.24%
  F1-Score:  98.25%
  Precision: 98.34%
  Recall:    98.24%

Per-Class F1-Scores:
  CNV     : 96.80%
  DME     : 99.79%
  DRUSEN  : 96.60%
  NORMAL  : 99.79%

Serial 05 | Seed 84 | SE_RESNET
Checkpoint: 05_se_resnet_seed84_epoch42_best_20260114_134347.pth
Epoch: 42 | Mode: best


...


Test Set Results:
  Accuracy:  98.35%
  F1-Score:  98.35%
  Precision: 98.43%
  Recall:    98.35%

Per-Class F1-Scores:
  CNV     : 96.79%
  DME     : 99.79%
  DRUSEN  : 96.80%
  NORMAL  : 100.00%

Serial 15 | Seed 126 | VIT_B16
Checkpoint: 15_vit_b16_seed126_epoch50_best_20260116_170850.pth
Epoch: 50 | Mode: best



Test Set Results:
  Accuracy:  98.55%
  F1-Score:  98.56%
  Precision: 98.63%
  Recall:    98.55%

Per-Class F1-Scores:
  CNV     : 97.19%
  DME     : 99.59%
  DRUSEN  : 97.46%
  NORMAL  : 100.00%

Serial 16 | Seed 3407 | VIT_B16
Checkpoint: 16_vit_b16_seed3407_epoch39_best_20260116_205856.pth
Epoch: 39 | Mode: best



Test Set Results:
  Accuracy:  98.24%
  F1-Score:  98.24%
  Precision: 98.36%
  Recall:    98.24%

Per-Class F1-Scores:
  CNV     : 96.61%
  DME     : 100.00%
  DRUSEN  : 96.36%
  NORMAL  : 100.00%

Serial 17 | Seed 3407 | RESNET_BASELINE
Checkpoint: 17_resnet_baseline_seed3407_epoch42_best_20260116_234021.pth
Epoch: 42 | Mode: best



Test Set Results:
  Accuracy:  98.76%
  F1-Score:  98.76%
  Precision: 98.78%
  Recall:    98.76%

Per-Class F1-Scores:
  CNV     : 97.97%
  DME     : 99.59%
  DRUSEN  : 97.91%
  NORMAL  : 99.59%

Serial 18 | Seed 3407 | CBAM_RESNET
Checkpoint: 18_cbam_resnet_seed3407_epoch40_best_20260117_020155.pth
Epoch: 40 | Mode: best



Test Set Results:
  Accuracy:  97.83%
  F1-Score:  97.84%
  Precision: 98.00%
  Recall:    97.83%

Per-Class F1-Scores:
  CNV     : 95.84%
  DME     : 98.96%
  DRUSEN  : 96.58%
  NORMAL  : 100.00%

EVALUATION COMPLETE


In [8]:
# RESULTS SUMMARY TABLE

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

# Create summary DataFrame
summary_data = []
for r in results:
    summary_data.append({
        'Serial': r['serial'],
        'Seed': r['seed'],
        'Model': r['model_name'],
        'Epoch': r['epoch'],
        'Accuracy': f"{r['accuracy']:.2f}%",
        'F1': f"{r['f1_macro']:.2f}%",
        'Precision': f"{r['precision_macro']:.2f}%",
        'Recall': f"{r['recall_macro']:.2f}%"
    })

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

print("\n" + "="*80)


RESULTS SUMMARY

  Serial  Seed           Model  Epoch Accuracy     F1 Precision Recall
      1    42 resnet_baseline     46   99.07% 99.07%    99.09% 99.07%
      2    42       se_resnet     47   98.24% 98.25%    98.34% 98.24%
      3    84 resnet_baseline     50   98.86% 98.87%    98.90% 98.86%
      4    42       se_resnet     47   98.24% 98.25%    98.34% 98.24%
      5    84       se_resnet     42   98.45% 98.45%    98.52% 98.45%
      6   126       se_resnet     48   98.35% 98.35%    98.42% 98.35%
      7    42     cbam_resnet     49   97.93% 97.94%    98.02% 97.93%
      8    84     cbam_resnet     33   98.55% 98.56%    98.62% 98.55%
      9   126     cbam_resnet     45   98.76% 98.76%    98.79% 98.76%
     10   126 resnet_baseline     39   99.07% 99.07%    99.08% 99.07%
     11  3407       se_resnet     42   98.04% 98.04%    98.18% 98.04%
     12    42         vit_b16     48   98.97% 98.97%    99.01% 98.97%
     13    84         vit_b16      4   97.42% 97.42%    97.51% 97.42%
 

In [9]:
# FILTER INCOMPLETE RUNS AND COMPUTE CORRECTED STATISTICS

import numpy as np
import pandas as pd

# Define results directory
results_dir = CHECKPOINT_DIR / "evaluation_results"
results_dir.mkdir(exist_ok=True)

print("\n" + "="*80)
print("FILTERING INCOMPLETE TRAINING RUNS")
print("="*80)

# Create DataFrame directly from results (not summary_data)
df_all = pd.DataFrame([{
    'Serial': r['serial'],
    'Seed': r['seed'],
    'Model': r['model_name'],
    'Epoch': r['epoch'],
    'Accuracy': r['accuracy'],  # Already numeric
    'F1': r['f1_macro'],  # Already numeric
    'Precision': r['precision_macro'],  # Already numeric
    'Recall': r['recall_macro']  # Already numeric
} for r in results])

# Identify incomplete runs (epoch < 30 suggests interrupted training)
INCOMPLETE_EPOCH_THRESHOLD = 30

incomplete_runs = df_all[df_all['Epoch'] < INCOMPLETE_EPOCH_THRESHOLD]
complete_runs = df_all[df_all['Epoch'] >= INCOMPLETE_EPOCH_THRESHOLD]

if len(incomplete_runs) > 0:
    print(f"\n⚠️  INCOMPLETE RUNS DETECTED (Epoch < {INCOMPLETE_EPOCH_THRESHOLD}):")
    print(incomplete_runs.to_string(index=False))
    print(f"\n✓ These {len(incomplete_runs)} run(s) will be EXCLUDED from statistics")
else:
    print(f"\n✓ No incomplete runs detected (all epochs >= {INCOMPLETE_EPOCH_THRESHOLD})")

print(f"\n✓ Complete runs: {len(complete_runs)}")
print("="*80)

# Data is already numeric - no conversion needed!
df_stats = complete_runs.copy()

# Compute statistics
summary = (
    df_stats.groupby("Model")
    .agg(
        Total_Runs=("Serial", "count"),
        Unique_Seeds=("Seed", pd.Series.nunique),
        Acc_Mean=("Accuracy", "mean"),
        Acc_Std=("Accuracy", "std"),
        Acc_Min=("Accuracy", "min"),
        Acc_Max=("Accuracy", "max"),
        F1_Mean=("F1", "mean"),
        F1_Std=("F1", "std")
    )
    .reset_index()
)

# Sort by Acc_Max BEFORE formatting (while still numeric)
summary = summary.sort_values(by="Acc_Max", ascending=False).reset_index(drop=True)

# Format to percentages AFTER sorting
def fmt_pct(x):
    return f"{x:.2f}%" if not np.isnan(x) else "N/A"

pct_cols = ["Acc_Mean", "Acc_Std", "Acc_Min", "Acc_Max", "F1_Mean", "F1_Std"]
for col in pct_cols:
    summary[col] = summary[col].apply(fmt_pct)

# Display corrected statistics
print("\n" + "="*80)
print("CORRECTED MULTI-SEED STATISTICS")
print("="*80)
print("(Incomplete runs excluded)")
print("\n" + summary.to_string(index=False))
print("\n" + "="*80)

# Save corrected statistics
corrected_csv = results_dir / "corrected_multi_seed_statistics.csv"
summary.to_csv(corrected_csv, index=False)
print(f"\n✅ Corrected statistics saved: {corrected_csv}")
print("="*80)



FILTERING INCOMPLETE TRAINING RUNS

⚠️  INCOMPLETE RUNS DETECTED (Epoch < 30):
 Serial  Seed   Model  Epoch  Accuracy        F1  Precision    Recall
     13    84 vit_b16      4 97.417355 97.417474   97.50562 97.417355

✓ These 1 run(s) will be EXCLUDED from statistics

✓ Complete runs: 17

CORRECTED MULTI-SEED STATISTICS
(Incomplete runs excluded)

          Model  Total_Runs  Unique_Seeds Acc_Mean Acc_Std Acc_Min Acc_Max F1_Mean F1_Std
resnet_baseline           4             4   98.94%   0.15%  98.76%  99.07%  98.94%  0.15%
        vit_b16           4             4   98.53%   0.32%  98.24%  98.97%  98.53%  0.32%
    cbam_resnet           4             4   98.27%   0.46%  97.83%  98.76%  98.28%  0.45%
      se_resnet           5             4   98.26%   0.15%  98.04%  98.45%  98.27%  0.15%


✅ Corrected statistics saved: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\corrected_multi_seed_statistics.csv


In [10]:
# GENERATE CONFUSION MATRICES AND ROC CURVES

from PIL import Image, ImageDraw, ImageFont
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import io

print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

# Create visualizations directory
viz_dir = results_dir / "visualizations"
viz_dir.mkdir(exist_ok=True)

def create_confusion_matrix_pil(cm, class_names, title, save_path):
    """
    Create confusion matrix visualization using PIL.
    
    Args:
        cm: Confusion matrix (numpy array)
        class_names: List of class names
        title: Title for the plot
        save_path: Path to save the image
    """
    n_classes = len(class_names)
    
    # Image dimensions
    cell_size = 100
    margin = 150
    width = margin + cell_size * n_classes + 100
    height = margin + cell_size * n_classes + 50
    
    # Create image
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    
    # Try to use a font, fall back to default if not available
    try:
        title_font = ImageFont.truetype("arial.ttf", 20)
        label_font = ImageFont.truetype("arial.ttf", 14)
        cell_font = ImageFont.truetype("arial.ttf", 16)
    except:
        title_font = ImageFont.load_default()
        label_font = ImageFont.load_default()
        cell_font = ImageFont.load_default()
    
    # Draw title
    draw.text((width // 2 - 100, 20), title, fill='black', font=title_font)
    
    # Normalize confusion matrix for color mapping
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Draw confusion matrix cells
    for i in range(n_classes):
        for j in range(n_classes):
            # Cell position
            x = margin + j * cell_size
            y = margin + i * cell_size
            
            # Cell color (green for correct, red for incorrect)
            if i == j:
                # Correct predictions - shades of green
                intensity = int(255 * (1 - cm_normalized[i, j]))
                color = (intensity, 255, intensity)
            else:
                # Incorrect predictions - shades of red
                if cm[i, j] > 0:
                    intensity = int(255 * (1 - min(cm_normalized[i, j] * 10, 1)))
                    color = (255, intensity, intensity)
                else:
                    color = (255, 255, 255)
            
            # Draw cell
            draw.rectangle([x, y, x + cell_size, y + cell_size], 
                          fill=color, outline='black', width=2)
            
            # Draw count
            count_text = str(cm[i, j])
            bbox = draw.textbbox((0, 0), count_text, font=cell_font)
            text_width = bbox[2] - bbox[0]
            text_height = bbox[3] - bbox[1]
            text_x = x + (cell_size - text_width) // 2
            text_y = y + (cell_size - text_height) // 2
            draw.text((text_x, text_y), count_text, fill='black', font=cell_font)
    
    # Draw row labels (True labels)
    draw.text((10, margin + n_classes * cell_size // 2), 
             "True", fill='black', font=label_font)
    for i, class_name in enumerate(class_names):
        y = margin + i * cell_size + cell_size // 2
        draw.text((margin - 80, y), class_name, fill='black', font=label_font)
    
    # Draw column labels (Predicted labels)
    draw.text((margin + n_classes * cell_size // 2, height - 30), 
             "Predicted", fill='black', font=label_font)
    for j, class_name in enumerate(class_names):
        x = margin + j * cell_size + 10
        draw.text((x, margin - 30), class_name, fill='black', font=label_font)
    
    # Save image
    img.save(save_path)
    print(f"  ✓ Saved: {save_path.name}")
    
    return img


def create_roc_curve_pil(y_true, y_pred_proba, class_names, title, save_path):
    """
    Create ROC curve visualization using PIL.
    
    Args:
        y_true: True labels
        y_pred_proba: Predicted probabilities (n_samples, n_classes)
        class_names: List of class names
        title: Title for the plot
        save_path: Path to save the image
    """
    n_classes = len(class_names)
    
    # Image dimensions
    width, height = 800, 600
    margin_left, margin_right = 80, 50
    margin_top, margin_bottom = 80, 80
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    
    # Create image
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    
    try:
        title_font = ImageFont.truetype("arial.ttf", 18)
        label_font = ImageFont.truetype("arial.ttf", 12)
    except:
        title_font = ImageFont.load_default()
        label_font = ImageFont.load_default()
    
    # Draw title
    draw.text((width // 2 - 100, 20), title, fill='black', font=title_font)
    
    # Draw axes
    draw.line([(margin_left, margin_top), 
              (margin_left, height - margin_bottom)], fill='black', width=2)
    draw.line([(margin_left, height - margin_bottom), 
              (width - margin_right, height - margin_bottom)], fill='black', width=2)
    
    # Draw diagonal (random classifier)
    draw.line([(margin_left, height - margin_bottom),
              (width - margin_right, margin_top)], fill='gray', width=1)
    
    # Binarize labels for one-vs-rest ROC
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    
    # Colors for each class
    colors = ['red', 'blue', 'green', 'orange']
    
    # Compute and draw ROC curve for each class
    for i, (class_name, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        
        # Convert to pixel coordinates
        points = []
        for fp, tp in zip(fpr, tpr):
            x = margin_left + int(fp * plot_width)
            y = height - margin_bottom - int(tp * plot_height)
            points.append((x, y))
        
        # Draw ROC curve
        if len(points) > 1:
            draw.line(points, fill=color, width=2)
        
        # Draw legend
        legend_y = margin_top + 20 + i * 20
        draw.line([(width - margin_right - 150, legend_y),
                  (width - margin_right - 130, legend_y)], fill=color, width=2)
        draw.text((width - margin_right - 120, legend_y - 10),
                 f"{class_name}: {roc_auc:.3f}", fill='black', font=label_font)
    
    # Draw axis labels
    draw.text((width // 2 - 60, height - 30), 
             "False Positive Rate", fill='black', font=label_font)
    draw.text((10, height // 2 - 10), 
             "True Positive Rate", fill='black', font=label_font)
    
    # Draw ticks
    for i in range(11):
        # X-axis ticks
        x = margin_left + int(i * plot_width / 10)
        draw.line([(x, height - margin_bottom), 
                  (x, height - margin_bottom + 5)], fill='black', width=1)
        draw.text((x - 10, height - margin_bottom + 10), 
                 f"{i/10:.1f}", fill='black', font=label_font)
        
        # Y-axis ticks
        y = height - margin_bottom - int(i * plot_height / 10)
        draw.line([(margin_left - 5, y), 
                  (margin_left, y)], fill='black', width=1)
        draw.text((margin_left - 30, y - 10), 
                 f"{i/10:.1f}", fill='black', font=label_font)
    
    # Save image
    img.save(save_path)
    print(f"  ✓ Saved: {save_path.name}")
    
    return img


# Generate visualizations for each complete model
print(f"\nGenerating confusion matrices and ROC curves...")
print(f"Output directory: {viz_dir}")
print()

for r in results:
    # Skip incomplete runs
    if r['epoch'] < INCOMPLETE_EPOCH_THRESHOLD:
        print(f"⊘ Skipping Serial {r['serial']:02d} (incomplete, epoch {r['epoch']})")
        continue
    
    serial = r['serial']
    seed = r['seed']
    model_name = r['model_name']
    
    print(f"\nSerial {serial:02d} | {model_name} | Seed {seed}")
    
    # Get confusion matrix
    cm = r['confusion_matrix']
    y_true = r['labels']
    y_pred = r['predictions']
    
    # Generate confusion matrix image
    cm_title = f"Serial {serial:02d} - {model_name.replace('_', ' ').title()} (Seed {seed})"
    cm_path = viz_dir / f"{serial:02d}_{model_name}_seed{seed}_confusion_matrix.png"
    create_confusion_matrix_pil(cm, CLASS_NAMES, cm_title, cm_path)
    
    # For ROC curve, we need predicted probabilities
    # Re-evaluate model to get probabilities
    info = all_checkpoints[serial][seed]
    best_checkpoint = find_best_checkpoint(all_checkpoints, serial, seed)
    
    model = create_model(model_name, NUM_CLASSES)
    checkpoint = torch.load(best_checkpoint['path'], map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE)
    model.eval()
    
    # Get predicted probabilities
    all_probs = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy())
    
    all_probs = np.array(all_probs)
    
    # Generate ROC curve image
    roc_title = f"Serial {serial:02d} - {model_name.replace('_', ' ').title()} (Seed {seed})"
    roc_path = viz_dir / f"{serial:02d}_{model_name}_seed{seed}_roc_curve.png"
    create_roc_curve_pil(y_true, all_probs, CLASS_NAMES, roc_title, roc_path)

print("\n" + "="*80)
print("VISUALIZATION COMPLETE")
print("="*80)
print(f"✓ Confusion matrices saved to: {viz_dir}")
print(f"✓ ROC curves saved to: {viz_dir}")
print("="*80)




GENERATING VISUALIZATIONS

Generating confusion matrices and ROC curves...
Output directory: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\visualizations


Serial 01 | resnet_baseline | Seed 42
  ✓ Saved: 01_resnet_baseline_seed42_confusion_matrix.png
  ✓ Saved: 01_resnet_baseline_seed42_roc_curve.png

Serial 02 | se_resnet | Seed 42
  ✓ Saved: 02_se_resnet_seed42_confusion_matrix.png
  ✓ Saved: 02_se_resnet_seed42_roc_curve.png

Serial 03 | resnet_baseline | Seed 84
  ✓ Saved: 03_resnet_baseline_seed84_confusion_matrix.png
  ✓ Saved: 03_resnet_baseline_seed84_roc_curve.png

Serial 04 | se_resnet | Seed 42
  ✓ Saved: 04_se_resnet_seed42_confusion_matrix.png
  ✓ Saved: 04_se_resnet_seed42_roc_curve.png

Serial 05 | se_resnet | Seed 84
  ✓ Saved: 05_se_resnet_seed84_confusion_matrix.png
  ✓ Saved: 05_se_resnet_seed84_roc_curve.png

Serial 06 | se_resnet | Seed 126
  ✓ Saved: 06_se_resnet_seed126_confusion_matrix.

In [11]:
# ENHANCED SUMMARY WITH ACCURACY SORTING (INCOMPLETE RUNS EXCLUDED)

import numpy as np
import pandas as pd

# Define results directory
results_dir = CHECKPOINT_DIR / "evaluation_results"
results_dir.mkdir(exist_ok=True)



# Print legend
print("\n" + "-"*80)
print("METRIC ABBREVIATIONS LEGEND:")
print("-"*80)
print("  Acc       : Accuracy (overall classification accuracy)")
print("  F1        : F1-Score (macro-averaged across all classes)")
print("  Prec      : Precision (macro-averaged)")
print("  Rec       : Recall (macro-averaged)")
print("  Epoch     : Training epoch number")
print("  Seed      : Random seed used for training")
print("  Serial    : Sequential checkpoint number")
print("  Mode      : Checkpoint mode from evaluation")
print("-"*80 + "\n")


print("\n" + "="*80)
print("ENHANCED STATISTICS (Sorted by Best Accuracy)")
print("="*80)
print("(Incomplete runs excluded)")
print()

if not results:
    print("No results to analyze")
else:
    # Build DataFrame from results
    summary_data = []
    for r in results:
        summary_data.append({
            'Serial': r['serial'],
            'Seed': r['seed'],
            'Model': r['model_name'],
            'Epoch': r['epoch'],
            'Accuracy': r['accuracy'],
            'F1': r['f1_macro'],
            'Precision': r['precision_macro'],
            'Recall': r['recall_macro']
        })
    
    df_all = pd.DataFrame(summary_data)
    
    # Filter out incomplete runs (same threshold as before)
    INCOMPLETE_EPOCH_THRESHOLD = 30
    df_stats = df_all[df_all['Epoch'] >= INCOMPLETE_EPOCH_THRESHOLD].copy()
    
    n_excluded = len(df_all) - len(df_stats)
    if n_excluded > 0:
        excluded_serials = df_all[df_all['Epoch'] < INCOMPLETE_EPOCH_THRESHOLD]['Serial'].tolist()
        print(f"⚠️  {n_excluded} incomplete run(s) excluded: Serial {excluded_serials}\n")
    
    # Group by model and compute statistics
    summary = (
        df_stats.groupby("Model")
        .agg(
            Runs=("Serial", "count"),
            Seeds=("Seed", pd.Series.nunique),
            Acc_Mean=("Accuracy", "mean"),
            Acc_Std=("Accuracy", "std"),
            Acc_Min=("Accuracy", "min"),
            Acc_Max=("Accuracy", "max"),
            F1_Mean=("F1", "mean"),
            F1_Std=("F1", "std")
        )
        .reset_index()
    )
    
    # Sort by best accuracy (while still numeric)
    summary = summary.sort_values(by="Acc_Max", ascending=False).reset_index(drop=True)
    
    # Format to percentages AFTER sorting
    def fmt_pct(x):
        return f"{x:.2f}%" if not np.isnan(x) else "N/A"
    
    pct_cols = ["Acc_Mean", "Acc_Std", "Acc_Min", "Acc_Max", "F1_Mean", "F1_Std"]
    for col in pct_cols:
        summary[col] = summary[col].apply(fmt_pct)
    
    # Display
    print(summary.to_string(index=False))
    print("\n" + "="*80)
    
    # Save to CSV
    csv_path = results_dir / "enhanced_statistics.csv"
    summary.to_csv(csv_path, index=False)
    print(f"\n✅ Enhanced statistics saved: {csv_path}")
    print("="*80)


--------------------------------------------------------------------------------
METRIC ABBREVIATIONS LEGEND:
--------------------------------------------------------------------------------
  Acc       : Accuracy (overall classification accuracy)
  F1        : F1-Score (macro-averaged across all classes)
  Prec      : Precision (macro-averaged)
  Rec       : Recall (macro-averaged)
  Epoch     : Training epoch number
  Seed      : Random seed used for training
  Serial    : Sequential checkpoint number
  Mode      : Checkpoint mode from evaluation
--------------------------------------------------------------------------------


ENHANCED STATISTICS (Sorted by Best Accuracy)
(Incomplete runs excluded)

⚠️  1 incomplete run(s) excluded: Serial [13]

          Model  Runs  Seeds Acc_Mean Acc_Std Acc_Min Acc_Max F1_Mean F1_Std
resnet_baseline     4      4   98.94%   0.15%  98.76%  99.07%  98.94%  0.15%
        vit_b16     4      4   98.53%   0.32%  98.24%  98.97%  98.53%  0.32%
    cbam_

In [12]:
# INDIVIDUAL MODEL RANKINGS - BEST MODEL AT TOP

import numpy as np
import pandas as pd

print("\n" + "="*80)
print("🏆 INDIVIDUAL MODEL RANKINGS (SORTED BY ACCURACY - BEST FIRST)")
print("="*80)
print("All individual runs ranked by test set accuracy")
print()

if not results:
    print("No results to analyze")
else:
    # Build detailed DataFrame
    individual_data = []
    for r in results:
        individual_data.append({
            'Model': r['model_name'],
            'Serial': r['serial'],
            'Seed': r['seed'],
            'Accuracy': r['accuracy'],
            'F1': r['f1_macro'],
            'Precision': r['precision_macro'],
            'Recall': r['recall_macro'],
            'Epoch': r['epoch']
        })
    
    df_individual = pd.DataFrame(individual_data)
    
    # Filter out incomplete runs (same threshold as before)
    INCOMPLETE_EPOCH_THRESHOLD = 30
    df_complete = df_individual[df_individual['Epoch'] >= INCOMPLETE_EPOCH_THRESHOLD].copy()
    
    n_excluded = len(df_individual) - len(df_complete)
    if n_excluded > 0:
        excluded_serials = df_individual[df_individual['Epoch'] < INCOMPLETE_EPOCH_THRESHOLD]['Serial'].tolist()
        print(f"⚠️  {n_excluded} incomplete run(s) excluded: Serial {excluded_serials}\n")
    
    # Sort by accuracy (best first)
    df_ranked = df_complete.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
    
    # Add rank column
    df_ranked.insert(0, 'Rank', range(1, len(df_ranked) + 1))
    
    # Format percentages
    def fmt_pct(x):
        return f"{x:.2f}%"
    
    # Create display dataframe with formatted values
    df_display = df_ranked.copy()
    df_display['Accuracy'] = df_display['Accuracy'].apply(fmt_pct)
    df_display['F1'] = df_display['F1'].apply(fmt_pct)
    df_display['Precision'] = df_display['Precision'].apply(fmt_pct)
    df_display['Recall'] = df_display['Recall'].apply(fmt_pct)
    
    # Display full ranking table
    print(df_display.to_string(index=False))
    
    # Highlight best model
    best = df_ranked.iloc[0]
    print("\n" + "="*80)
    print("🥇 BEST MODEL (Highest Test Set Accuracy)")
    print("="*80)
    print(f"  Model:      {best['Model']}")
    print(f"  Serial:     {best['Serial']:02d}")
    print(f"  Seed:       {best['Seed']}")
    print(f"  Accuracy:   {best['Accuracy']:.2f}%")
    print(f"  F1-Score:   {best['F1']:.2f}%")
    print(f"  Precision:  {best['Precision']:.2f}%")
    print(f"  Recall:     {best['Recall']:.2f}%")
    print(f"  Epoch:      {best['Epoch']}")
    print("="*80)
    
    # Show top 5
    print("\n" + "="*80)
    print("🏅 TOP 5 MODELS")
    print("="*80)
    top5 = df_display.head(5)[['Rank', 'Model', 'Serial', 'Seed', 'Accuracy', 'F1']]
    print(top5.to_string(index=False))
    print("="*80)
    
    # Save individual rankings to CSV
    results_dir = CHECKPOINT_DIR / "evaluation_results"
    individual_csv = results_dir / "individual_model_rankings.csv"
    
    # Save with numeric values for further analysis
    df_ranked.to_csv(individual_csv, index=False)
    print(f"\n✅ Individual rankings saved: {individual_csv}")
    print("="*80)


🏆 INDIVIDUAL MODEL RANKINGS (SORTED BY ACCURACY - BEST FIRST)
All individual runs ranked by test set accuracy

⚠️  1 incomplete run(s) excluded: Serial [13]

 Rank           Model  Serial  Seed Accuracy     F1 Precision Recall  Epoch
    1 resnet_baseline       1    42   99.07% 99.07%    99.09% 99.07%     46
    2 resnet_baseline      10   126   99.07% 99.07%    99.08% 99.07%     39
    3         vit_b16      12    42   98.97% 98.97%    99.01% 98.97%     48
    4 resnet_baseline       3    84   98.86% 98.87%    98.90% 98.86%     50
    5     cbam_resnet       9   126   98.76% 98.76%    98.79% 98.76%     45
    6 resnet_baseline      17  3407   98.76% 98.76%    98.78% 98.76%     42
    7         vit_b16      15   126   98.55% 98.56%    98.63% 98.55%     50
    8     cbam_resnet       8    84   98.55% 98.56%    98.62% 98.55%     33
    9       se_resnet       5    84   98.45% 98.45%    98.52% 98.45%     42
   10         vit_b16      14    84   98.35% 98.35%    98.43% 98.35%     48
   11

In [13]:
# SAVE RESULTS

results_dir = CHECKPOINT_DIR / "evaluation_results"
results_dir.mkdir(exist_ok=True)

# MULTI-SEED STATISTICS

print("\n" + "="*80)
print("MULTI-SEED STATISTICS")
print("="*80)

# Group by model type
model_stats = defaultdict(lambda: {'accuracy': [], 'f1': []})

for r in results:
    model_stats[r['model_name']]['accuracy'].append(r['accuracy'])
    model_stats[r['model_name']]['f1'].append(r['f1_macro'])

stats_data = []
for model_name in sorted(model_stats.keys()):
    acc_values = model_stats[model_name]['accuracy']
    f1_values = model_stats[model_name]['f1']
    
    if len(acc_values) > 1:
        stats_data.append({
            'Model': model_name,
            'Seeds': len(acc_values),
            'Acc Mean': f"{np.mean(acc_values):.2f}%",
            'Acc Std': f"{np.std(acc_values):.2f}%",
            'F1 Mean': f"{np.mean(f1_values):.2f}%",
            'F1 Std': f"{np.std(f1_values):.2f}%"
        })
    else:
        stats_data.append({
            'Model': model_name,
            'Seeds': len(acc_values),
            'Acc Mean': f"{acc_values[0]:.2f}%",
            'Acc Std': 'N/A',
            'F1 Mean': f"{f1_values[0]:.2f}%",
            'F1 Std': 'N/A'
        })

if stats_data:
    stats_df = pd.DataFrame(stats_data)
    print("\n", stats_df.to_string(index=False))


print("\n" + "="*80)

# Save summary
summary_df.to_csv(results_dir / "test_results_summary.csv", index=False)
print(f"\n✅ Summary saved: {results_dir / 'test_results_summary.csv'}")

# Save statistics
if stats_data:
    stats_df.to_csv(results_dir / "multi_seed_statistics.csv", index=False)
    print(f"✅ Statistics saved: {results_dir / 'multi_seed_statistics.csv'}")

print("\n" + "="*80)
print("EVALUATION COMPLETE - ALL RESULTS SAVED")
print("="*80)


MULTI-SEED STATISTICS

           Model  Seeds Acc Mean Acc Std F1 Mean F1 Std
    cbam_resnet      4   98.27%   0.40%  98.28%  0.39%
resnet_baseline      4   98.94%   0.13%  98.94%  0.13%
      se_resnet      5   98.26%   0.14%  98.27%  0.14%
        vit_b16      5   98.31%   0.51%  98.31%  0.51%


✅ Summary saved: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\test_results_summary.csv
✅ Statistics saved: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints\evaluation_results\multi_seed_statistics.csv

EVALUATION COMPLETE - ALL RESULTS SAVED


In [14]:
# ============================================================================
# EXTRACT PER-CLASS METRICS FOR TABLE 5.2.2 (BEST 4 RUNS)
# Uses existing checkpoint discovery infrastructure
# ============================================================================

from sklearn.metrics import classification_report
import numpy as np

print("="*80)
print("EXTRACTING PER-CLASS METRICS FOR TABLE 5.2.2")
print("="*80)

# The 4 best runs we need (from your results)
BEST_RUNS_FOR_TABLE = [
    {'serial': 1, 'seed': 42, 'name': 'ResNet-50'},
    {'serial': 5, 'seed': 84, 'name': 'SE-ResNet50'},
    {'serial': 9, 'seed': 126, 'name': 'CBAM-ResNet50'},
    {'serial': 12, 'seed': 42, 'name': 'ViT-B/16'},
]

# Function to evaluate and get per-class metrics
def get_per_class_metrics(model, data_loader, device):
    """Evaluate model and return detailed per-class metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    
    # Generate classification report
    report = classification_report(
        y_true, y_pred,
        target_names=['CNV', 'DME', 'DRUSEN', 'NORMAL'],
        output_dict=True,
        digits=4
    )
    
    return report

# Process each best run
per_class_results = []

for run_info in BEST_RUNS_FOR_TABLE:
    serial = run_info['serial']
    seed = run_info['seed']
    name = run_info['name']
    
    print(f"\n{'='*80}")
    print(f"Processing: {name} (Serial {serial}, Seed {seed})")
    print(f"{'='*80}")
    
    # Find best checkpoint using your existing function
    best_checkpoint = find_best_checkpoint(all_checkpoints, serial, seed)
    
    if not best_checkpoint:
        print(f"❌ ERROR: No checkpoint found for Serial {serial}, Seed {seed}")
        continue
    
    checkpoint_path = best_checkpoint['path']
    model_name = all_checkpoints[serial][seed]['model_name']
    
    print(f"Checkpoint: {checkpoint_path.name}")
    print(f"Model type: {model_name}")
    print(f"Epoch: {best_checkpoint['epoch']}")
    
    # Load model architecture using the create_model factory function (defined in Cell 3)
    try:
        model = create_model(model_name, num_classes=NUM_CLASSES)
        print(f"✅ Model architecture created: {model_name}")
    except Exception as e:
        print(f"❌ ERROR creating model architecture: {e}")
        continue
    
    # Load checkpoint weights
    try:
        checkpoint_data = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint_data['model_state_dict'])
        model = model.to(DEVICE)
        print("✅ Model loaded successfully")
    except Exception as e:
        print(f"❌ ERROR loading checkpoint: {e}")
        continue
    
    # Get per-class metrics
    try:
        report = get_per_class_metrics(model, test_loader, DEVICE)
        
        # Print formatted table
        print(f"\n{name} (Serial {serial}, Seed {seed}, Epoch {best_checkpoint['epoch']})")
        print(f"Overall Accuracy: {report['accuracy']*100:.2f}%")
        print("-" * 75)
        print(f"{'Class':<10} | {'Precision':>10} | {'Recall':>10} | {'F1-Score':>10}")
        print("-" * 75)
        
        for class_name in ['CNV', 'DME', 'DRUSEN', 'NORMAL']:
            p = report[class_name]['precision'] * 100
            r = report[class_name]['recall'] * 100
            f1 = report[class_name]['f1-score'] * 100
            print(f"{class_name:<10} | {p:>9.2f}% | {r:>9.2f}% | {f1:>9.2f}%")
        
        print("-" * 75)
        macro_p = report['macro avg']['precision'] * 100
        macro_r = report['macro avg']['recall'] * 100
        macro_f1 = report['macro avg']['f1-score'] * 100
        print(f"{'Macro':<10} | {macro_p:>9.2f}% | {macro_r:>9.2f}% | {macro_f1:>9.2f}%")
        
        # Store results
        per_class_results.append({
            'name': name,
            'serial': serial,
            'seed': seed,
            'epoch': best_checkpoint['epoch'],
            'report': report
        })
        
        print("✅ Evaluation complete")
        
    except Exception as e:
        print(f"❌ ERROR during evaluation: {e}")
        continue

# ============================================================================
# SAVE RESULTS TO CSV
# ============================================================================

if len(per_class_results) == 4:
    print("\n" + "="*80)
    print("SAVING PER-CLASS METRICS TO CSV")
    print("="*80)
    
    import pandas as pd
    
    # Create rows for CSV
    rows = []
    for result in per_class_results:
        report = result['report']
        
        # Add per-class metrics
        for class_name in ['CNV', 'DME', 'DRUSEN', 'NORMAL']:
            rows.append({
                'Model': result['name'],
                'Serial': result['serial'],
                'Seed': result['seed'],
                'Epoch': result['epoch'],
                'Class': class_name,
                'Precision': f"{report[class_name]['precision']*100:.2f}%",
                'Recall': f"{report[class_name]['recall']*100:.2f}%",
                'F1-Score': f"{report[class_name]['f1-score']*100:.2f}%"
            })
        
        # Add macro average
        rows.append({
            'Model': result['name'],
            'Serial': result['serial'],
            'Seed': result['seed'],
            'Epoch': result['epoch'],
            'Class': 'Macro Avg',
            'Precision': f"{report['macro avg']['precision']*100:.2f}%",
            'Recall': f"{report['macro avg']['recall']*100:.2f}%",
            'F1-Score': f"{report['macro avg']['f1-score']*100:.2f}%"
        })
    
    # Save to CSV
    df = pd.DataFrame(rows)
    output_file = CHECKPOINT_DIR.parent / 'per_class_metrics_table_5_2_2.csv'
    df.to_csv(output_file, index=False)
    
    print(f"✅ Saved to: {output_file}")
    print(f"   {len(rows)} rows written")
    
    # Display the table
    print("\n" + "="*80)
    print("COMPLETE TABLE 5.2.2 DATA")
    print("="*80)
    print(df.to_string(index=False))
    
else:
    print("\n" + "="*80)
    print(f"⚠️  WARNING: Only {len(per_class_results)}/4 models processed")
    print("="*80)

print("\n" + "="*80)
print("EXTRACTION COMPLETE - COPY THESE NUMBERS TO DISSERTATION TABLE 5.2.2")
print("="*80)

EXTRACTING PER-CLASS METRICS FOR TABLE 5.2.2

Processing: ResNet-50 (Serial 1, Seed 42)
Checkpoint: 01_resnet_baseline_seed42_epoch46_best_20260114_001728.pth
Model type: resnet_baseline
Epoch: 46
✅ Model architecture created: resnet_baseline
✅ Model loaded successfully

ResNet-50 (Serial 1, Seed 42, Epoch 46)
Overall Accuracy: 99.07%
---------------------------------------------------------------------------
Class      |  Precision |     Recall |   F1-Score
---------------------------------------------------------------------------
CNV        |     97.18% |     99.59% |     98.37%
DME        |     99.59% |    100.00% |     99.79%
DRUSEN     |     99.58% |     97.11% |     98.33%
NORMAL     |    100.00% |     99.59% |     99.79%
---------------------------------------------------------------------------
Macro      |     99.09% |     99.07% |     99.07%
✅ Evaluation complete

Processing: SE-ResNet50 (Serial 5, Seed 84)
Checkpoint: 05_se_resnet_seed84_epoch42_best_20260114_134347.pth
Mod